In [15]:
import os
import zipfile
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, Dropout

In [16]:
INPUT_DIR = '/kaggle/input/competitions/data-science-bowl-2018/'
WORK_DIR = '/kaggle/working/'
TRAIN_PATH = WORK_DIR + 'train/'
TEST_PATH = WORK_DIR + 'test/'

IMG_WIDTH, IMG_HEIGHT, IMG_CHANNELS = 128, 128, 3

print("Extracting datasets...")
with zipfile.ZipFile('/kaggle/input/competitions/data-science-bowl-2018/stage1_train.zip', 'r') as zip_ref:
    zip_ref.extractall(TRAIN_PATH)
with zipfile.ZipFile('/kaggle/input/competitions/data-science-bowl-2018/stage1_test.zip', 'r') as zip_ref:
    zip_ref.extractall(TEST_PATH)
print("Extraction complete!")

train_ids = next(os.walk(TRAIN_PATH))[1]
test_ids = next(os.walk(TEST_PATH))[1]

Extracting datasets...
Extraction complete!


In [17]:
X_train = np.zeros((len(train_ids), IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.uint8)
Y_train = np.zeros((len(train_ids), IMG_HEIGHT, IMG_WIDTH, 1), dtype=np.bool_)

for n, id_ in enumerate(train_ids):
    path = TRAIN_PATH + id_
    img = cv2.imread(path + '/images/' + id_ + '.png', cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    X_train[n] = img
    
    # Read and combine masks
    mask = np.zeros((IMG_HEIGHT, IMG_WIDTH, 1), dtype=np.bool_)
    for mask_file in next(os.walk(path + '/masks/'))[2]:
        mask_ = cv2.imread(path + '/masks/' + mask_file, cv2.IMREAD_GRAYSCALE)
        mask_ = cv2.resize(mask_, (IMG_WIDTH, IMG_HEIGHT))
        mask_ = np.expand_dims(mask_, axis=-1)
        mask = np.maximum(mask, mask_ > 0) # Combine all nuclei into one mask
    Y_train[n] = mask

X_test = np.zeros((len(test_ids), IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype=np.uint8)
sizes_test = [] 
for n, id_ in enumerate(test_ids):
    path = TEST_PATH + id_
    img = cv2.imread(path + '/images/' + id_ + '.png', cv2.IMREAD_COLOR)
    sizes_test.append([img.shape[0], img.shape[1]])
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    X_test[n] = img

In [18]:
def build_unet(input_shape=(128, 128, 3)):
    inputs = Input(input_shape)
    s = tf.keras.layers.Lambda(lambda x: x / 255.0)(inputs) # Normalize

    # Encoder
    c1 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(s)
    c1 = Dropout(0.1)(c1)
    c1 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p1)
    c2 = Dropout(0.1)(c2)
    c2 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p2)
    c3 = Dropout(0.2)(c3)
    c3 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c3)
    p3 = MaxPooling2D((2, 2))(c3)

    # Bottleneck
    c4 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p3)
    c4 = Dropout(0.2)(c4)
    c4 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c4)

    # Decoder
    u5 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = concatenate([u5, c3])
    c5 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u5)
    c5 = Dropout(0.2)(c5)
    c5 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c5)

    u6 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = concatenate([u6, c2])
    c6 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u6)
    c6 = Dropout(0.1)(c6)
    c6 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c6)

    u7 = Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = concatenate([u7, c1])
    c7 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u7)
    c7 = Dropout(0.1)(c7)
    c7 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c7)

    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c7)
    return Model(inputs=[inputs], outputs=[outputs])

In [19]:
model = build_unet(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("training model")
# setting verbose=0
model.fit(
    X_train, 
    Y_train, 
    validation_split=0.2, 
    batch_size=16, 
    epochs=25, 
    verbose=2,
    callbacks=[EpochStatsCallback()] 
)

training model
Epoch 1/25
01 | Train Loss: 0.5048 - Train Acc: 0.8026 | Val Loss: 0.3614 - Val Acc: 0.8428
34/34 - 25s - 728ms/step - accuracy: 0.8026 - loss: 0.5048 - val_accuracy: 0.8428 - val_loss: 0.3614
Epoch 2/25
02 | Train Loss: 0.2743 - Train Acc: 0.8483 | Val Loss: 0.2190 - Val Acc: 0.8428
34/34 - 1s - 42ms/step - accuracy: 0.8483 - loss: 0.2743 - val_accuracy: 0.8428 - val_loss: 0.2190
Epoch 3/25
03 | Train Loss: 0.2002 - Train Acc: 0.8630 | Val Loss: 0.1753 - Val Acc: 0.9198
34/34 - 1s - 39ms/step - accuracy: 0.8630 - loss: 0.2002 - val_accuracy: 0.9198 - val_loss: 0.1753
Epoch 4/25
04 | Train Loss: 0.1704 - Train Acc: 0.9335 | Val Loss: 0.1507 - Val Acc: 0.9461
34/34 - 1s - 39ms/step - accuracy: 0.9335 - loss: 0.1704 - val_accuracy: 0.9461 - val_loss: 0.1507
Epoch 5/25
05 | Train Loss: 0.1391 - Train Acc: 0.9494 | Val Loss: 0.1165 - Val Acc: 0.9597
34/34 - 1s - 39ms/step - accuracy: 0.9494 - loss: 0.1391 - val_accuracy: 0.9597 - val_loss: 0.1165
Epoch 6/25
06 | Train Loss: 

In [20]:
def rle_encoding(x):
    dots = np.where(x.T.flatten() == 1)[0]
    run_lengths = []
    prev = -2
    for b in dots:
        if (b > prev+1): run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return " ".join([str(i) for i in run_lengths])

print("predicting on test data")
preds_test = model.predict(X_test)
preds_test_t = (preds_test > 0.5).astype(np.uint8)

new_test_ids = []
rles = []

print("processing masks and generating Run-Length Encoding")
for n, id_ in enumerate(test_ids):
    orig_h, orig_w = sizes_test[n][0], sizes_test[n][1]
    
    # Resize prediction back to original image size
    pred_resized = cv2.resize(preds_test_t[n], (orig_w, orig_h))
    
    # Find distinct nuclei using OpenCV
    num_labels, labels = cv2.connectedComponents(pred_resized)

    if num_labels == 1:
        new_test_ids.append(id_)
        rles.append('')
        continue
        
    for i in range(1, num_labels):
        mask = (labels == i).astype(np.uint8)
        rles.append(rle_encoding(mask))
        new_test_ids.append(id_)
sub = pd.DataFrame({'ImageId': new_test_ids, 'EncodedPixels': rles})
sub.to_csv('submission.csv', index=False)
print("submission created successfully!")

predicting on test data
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 381ms/step
processing masks and generating Run-Length Encoding
submission created successfully!
